<img height="100" src="https://i.postimg.cc/gjptBxF4/logo-gas-removebg-preview.jpg" width="250"/>

# Análise de cluster por metodo Hierárquico - Linkage Completo

Govender, P., and V. Sivakumar. 2020. “Application of K-Means and Hierarchical Clustering Techniques for Analysis of Air Pollution: A Review (1980–2019).” Atmospheric Pollution Research 11 (1): 40–56. https://doi.org/10.1016/j.apr.2019.09.009.

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import os
import glob
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt
from tqdm import tqdm

# Configurações
diretorio_completo = os.path.join(os.getcwd(), 'inputs', 'data', 'completo')
arquivos_nc = glob.glob(os.path.join(diretorio_completo, "point_id_*.nc"))
variaveis_clima = ['ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'T2M_MAX', 'T2M_MIN', 'RH2M', 'WS2M']

print(f"Total de arquivos: {len(arquivos_nc)}")

# Extrair estatísticas de cada arquivo
dados_agregados = []

for arquivo in tqdm(arquivos_nc, desc="Processando arquivos"):
    ponto_id = os.path.basename(arquivo).replace('point_id_', '').replace('.nc', '')

    try:
        with xr.open_dataset(arquivo) as ds:
            estatisticas = {'ID': ponto_id}

            for var in variaveis_clima:
                if var in ds.data_vars:
                    estatisticas[f'{var}_mean'] =   float(ds[var].mean(skipna=True).values)
                    estatisticas[f'{var}_std'] =    float(ds[var].std(skipna=True).values)
                    estatisticas[f'{var}_median'] = float(ds[var].median(skipna=True).values)
                    estatisticas[f'{var}_min'] =    float(ds[var].min(skipna=True).values)
                    estatisticas[f'{var}_max'] =    float(ds[var].max(skipna=True).values)

            dados_agregados.append(estatisticas)

    except Exception as e:
        print(f"Erro ao processar {ponto_id}: {e}")

# Criar DataFrame
df_cluster = pd.DataFrame(dados_agregados)
df_cluster = df_cluster.set_index('ID')

print(f"\nDados agregados: {df_cluster.shape}")
print(df_cluster.head())

# Remover linhas com NaN (se houver)
df_cluster_clean = df_cluster.dropna()
print(f"Após remover NaN: {df_cluster_clean.shape}")

# Padronizar os dados
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster_clean)

# Clustering Hierárquico
# Métodos de linkage: 'ward', 'complete', 'average', 'single'
# Ward é robusto a outliers
linkage_matrix = linkage(X_scaled, method='ward')

# Definir número de clusters (ajuste conforme necessário)
n_clusters = 5

# Aplicar clustering
model = AgglomerativeClustering(n_clusters=n_clusters, metric="euclidean",
        memory=None,
        connectivity=None,
        compute_full_tree="auto",
        linkage="ward",
        distance_threshold=None,
        compute_distances=True)

clusters = model.fit_predict(X_scaled)

# Adicionar clusters ao DataFrame
df_cluster_clean['Cluster'] = clusters

# Salvar resultados
resultado_path = os.path.join(diretorio_completo, 'resultados_cluster.csv')
df_cluster_clean.to_csv(resultado_path)

print(f"\n{'='*80}")
print("RESULTADOS DO CLUSTERING")
print(f"{'='*80}")
print(f"\nDistribuição dos clusters:")
print(df_cluster_clean['Cluster'].value_counts().sort_index())
print(f"\nResultados salvos em: {resultado_path}")

In [ ]:
# Análise das características de cada cluster
print("\n" + "="*100)
print("CARACTERIZAÇÃO DOS CLUSTERS")
print("="*100)

for i in range(n_clusters):
    cluster_data = df_cluster_clean[df_cluster_clean['Cluster'] == i]

    print(f"\n{'='*100}")
    print(f"CLUSTER {i} - {len(cluster_data)} pontos ({len(cluster_data)/len(df_cluster_clean)*100:.1f}%)")
    print(f"{'='*100}")

    # Médias das variáveis originais (não padronizadas)
    print("\nMédias das variáveis climáticas:")
    for var in variaveis_clima:
        mean_col = f'{var}_mean'
        if mean_col in cluster_data.columns:
            print(f"  {var:20s}: {cluster_data[mean_col].mean():>10.2f} (±{cluster_data[mean_col].std():>8.2f})")

    # IDs dos primeiros pontos
    print(f"\nPrimeiros IDs: {', '.join(cluster_data.index[:5].tolist())}")

In [ ]:
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, fcluster

sns.set_theme(style='white', context='paper', font_scale=1.2, palette='tab10')

# Calcular o threshold para 6 clusters
# Pega a distância da 6ª fusão a partir do final
threshold = linkage_matrix[-5, 2]

plt.figure(figsize=(10, 20))
dendrogram(
    linkage_matrix,
    labels=df_cluster_clean.index.values,
    orientation='left',
    leaf_font_size=5,
    color_threshold=threshold,  # ← Colorir 6 clusters
    above_threshold_color='gray'  # ← Fusões acima do threshold em cinza
)

# Adicionar linha horizontal mostrando o corte
plt.axvline(x=threshold, color='red', linestyle='--', linewidth=2,
            label=f'Corte para 5 clusters (d={threshold:.2f})')

plt.title('Dendrograma - 5 Clusters', fontsize=14, fontweight='bold')
plt.xlabel('Distância')
plt.ylabel('ID dos Pontos')
plt.legend(loc='upper right')
plt.tight_layout()
# plt.savefig(os.path.join(os.getcwd(), 'dendrograma_6_clusters.jpg'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Comparação visual dos clusters
import seaborn as sns

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, var in enumerate(variaveis_clima):
    ax = axes[idx]

    # Dados para boxplot
    data_plot = []
    labels_plot = []

    for i in range(n_clusters):
        cluster_data = df_cluster_clean[df_cluster_clean['Cluster'] == i]
        data_plot.append(cluster_data[f'{var}_mean'].values)
        labels_plot.append(f'C{i}')

    bp = ax.boxplot(data_plot, tick_labels=labels_plot, patch_artist=True)

    # Colorir
    colors = plt.cm.Set3(range(n_clusters))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)

    ax.set_title(var, fontsize=12, fontweight='bold')
    ax.set_xlabel('Cluster')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
#plt.savefig(os.path.join(diretorio_completo, 'comparacao_clusters.jpg'), dpi=300)
plt.show()

# Identificando número adequado de clusters

Análise do dendograma

In [ ]:
# Analisar distâncias de fusão
from scipy.cluster.hierarchy import fcluster

# Últimas fusões (maiores distâncias)
last_merges = linkage_matrix[-10:, 2]
print("Distâncias das últimas 10 fusões:")
for i, dist in enumerate(last_merges, 1):
    print(f"{i} clusters: {dist:.2f}")

# Plotar cotovelo das distâncias
plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), last_merges[::-1], 'bo-')
plt.xlabel('Número de Clusters')
plt.ylabel('Distância de Fusão')
plt.title('Método do Cotovelo - Distâncias de Fusão')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Validar a escolha com outras métricas
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Testar range de 4 a 10 clusters
for k in range(4, 11):
    labels = fcluster(linkage_matrix, k, criterion='maxclust')
    silhouette = silhouette_score(X_scaled, labels)
    db_index = davies_bouldin_score(X_scaled, labels)
    print(f"K={k}: Silhouette={silhouette:.3f}, DB Index={db_index:.3f}")

Silhouette Score

In [ ]:
from sklearn.metrics import silhouette_score, silhouette_samples

silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    model = AgglomerativeClustering(n_clusters=k, linkage='ward')
    clusters = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, clusters)
    silhouette_scores.append(score)
    print(f"{k} clusters: Silhouette = {score:.4f}")

# Plotar
plt.figure(figsize=(10, 6))
plt.plot(K_range, silhouette_scores, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Número de Clusters')
plt.ylabel('Silhouette Score')
plt.title('Análise do Silhouette Score')
plt.grid(True, alpha=0.3)
plt.axhline(y=max(silhouette_scores), color='r', linestyle='--', alpha=0.5,
            label=f'Máximo: {max(silhouette_scores):.4f}')
plt.legend()
plt.tight_layout()
plt.show()

# Melhor número de clusters
melhor_k = K_range[silhouette_scores.index(max(silhouette_scores))]
print(f"\nMelhor número de clusters: {melhor_k} (Silhouette = {max(silhouette_scores):.4f})")

Davies-Bouldin Index

In [ ]:
from sklearn.metrics import davies_bouldin_score

db_scores = []

for k in K_range:
    model = AgglomerativeClustering(n_clusters=k, linkage='ward')
    clusters = model.fit_predict(X_scaled)
    score = davies_bouldin_score(X_scaled, clusters)
    db_scores.append(score)
    print(f"{k} clusters: Davies-Bouldin = {score:.4f}")

# Plotar
plt.figure(figsize=(10, 6))
plt.plot(K_range, db_scores, 'ro-', linewidth=2, markersize=8)
plt.xlabel('Número de Clusters')
plt.ylabel('Davies-Bouldin Index')
plt.title('Análise do Davies-Bouldin Index (menor é melhor)')
plt.grid(True, alpha=0.3)
plt.axhline(y=min(db_scores), color='b', linestyle='--', alpha=0.5,
            label=f'Mínimo: {min(db_scores):.4f}')
plt.legend()
plt.tight_layout()
plt.show()

melhor_k_db = K_range[db_scores.index(min(db_scores))]
print(f"\nMelhor número de clusters: {melhor_k_db} (DB = {min(db_scores):.4f})")

Calinski-Harabasz Index

In [ ]:
from sklearn.metrics import calinski_harabasz_score

ch_scores = []

for k in K_range:
    model = AgglomerativeClustering(n_clusters=k, linkage='ward')
    clusters = model.fit_predict(X_scaled)
    score = calinski_harabasz_score(X_scaled, clusters)
    ch_scores.append(score)
    print(f"{k} clusters: Calinski-Harabasz = {score:.2f}")

# Plotar
plt.figure(figsize=(10, 6))
plt.plot(K_range, ch_scores, 'go-', linewidth=2, markersize=8)
plt.xlabel('Número de Clusters')
plt.ylabel('Calinski-Harabasz Index')
plt.title('Análise do Calinski-Harabasz Index (maior é melhor)')
plt.grid(True, alpha=0.3)
plt.axhline(y=max(ch_scores), color='r', linestyle='--', alpha=0.5,
            label=f'Máximo: {max(ch_scores):.2f}')
plt.legend()
plt.tight_layout()
plt.show()

Comparação Consolidada

In [ ]:
# Painel com todos os métodos
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Silhouette
axes[0, 0].plot(K_range, silhouette_scores, 'bo-', linewidth=2, markersize=8)
axes[0, 0].set_title('Silhouette Score (↑)', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Número de Clusters')
axes[0, 0].grid(True, alpha=0.3)

# Davies-Bouldin
axes[0, 1].plot(K_range, db_scores, 'ro-', linewidth=2, markersize=8)
axes[0, 1].set_title('Davies-Bouldin Index (↓)', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Número de Clusters')
axes[0, 1].grid(True, alpha=0.3)

# Calinski-Harabasz
axes[1, 0].plot(K_range, ch_scores, 'go-', linewidth=2, markersize=8)
axes[1, 0].set_title('Calinski-Harabasz Index (↑)', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Número de Clusters')
axes[1, 0].grid(True, alpha=0.3)

# Distâncias de fusão
axes[1, 1].plot(range(2, 12), linkage_matrix[-10:, 2][::-1], 'mo-', linewidth=2, markersize=8)
axes[1, 1].set_title('Distâncias de Fusão (Cotovelo)', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Número de Clusters')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
#plt.savefig(os.path.join(diretorio_completo, 'analise_numero_clusters.jpg'), dpi=300)
plt.show()

# Resumo
print(f"\n{'='*80}")
print("RECOMENDAÇÕES DE NÚMERO DE CLUSTERS")
print(f"{'='*80}")
print(f"Silhouette Score:      {K_range[silhouette_scores.index(max(silhouette_scores))]} clusters")
print(f"Davies-Bouldin Index:  {K_range[db_scores.index(min(db_scores))]} clusters")
print(f"Calinski-Harabasz:     {K_range[ch_scores.index(max(ch_scores))]} clusters")

In [ ]:
# Análise comparativa detalhada entre 2, 5 e 8 clusters
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_samples

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, n_clust in enumerate([2, 5, 8]):
    model = AgglomerativeClustering(n_clusters=n_clust, linkage='ward')
    cluster_labels = model.fit_predict(X_scaled)

    # Silhouette por amostra
    silhouette_vals = silhouette_samples(X_scaled, cluster_labels)

    ax = axes[idx]
    y_lower = 10

    for i in range(n_clust):
        cluster_silhouette_vals = silhouette_vals[cluster_labels == i]
        cluster_silhouette_vals.sort()

        size_cluster = cluster_silhouette_vals.shape[0]
        y_upper = y_lower + size_cluster

        color = plt.cm.Set3(i / n_clust)
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_silhouette_vals,
                         facecolor=color, edgecolor=color, alpha=0.7)

        ax.text(-0.05, y_lower + 0.5 * size_cluster, str(i))
        y_lower = y_upper + 10

    ax.set_title(f'{n_clust} Clusters\nSilhouette: {silhouette_score(X_scaled, cluster_labels):.4f}')
    ax.set_xlabel('Silhouette Score')
    ax.set_ylabel('Cluster')
    ax.axvline(x=silhouette_score(X_scaled, cluster_labels), color="red", linestyle="--")
    ax.set_xlim([-0.1, 0.6])

plt.tight_layout()
#plt.savefig(os.path.join(diretorio_completo, 'comparacao_silhouette.jpg'), dpi=300)
plt.show()

In [ ]:
# Recalcular com 5 clusters
n_clusters_final = 5

model_final = AgglomerativeClustering(n_clusters=n_clusters_final, linkage='ward')
df_cluster_clean['Cluster_Final'] = model_final.fit_predict(X_scaled)

print(f"\n{'='*100}")
print(f"ANÁLISE FINAL COM {n_clusters_final} CLUSTERS")
print(f"{'='*100}")

print(f"\nDistribuição:")
print(df_cluster_clean['Cluster_Final'].value_counts().sort_index())

print(f"\n{'='*100}")
print("CARACTERIZAÇÃO DOS CLUSTERS")
print(f"{'='*100}")

for i in range(n_clusters_final):
    cluster_data = df_cluster_clean[df_cluster_clean['Cluster_Final'] == i]

    print(f"\n{'='*100}")
    print(f"CLUSTER {i} - {len(cluster_data)} pontos ({len(cluster_data)/len(df_cluster_clean)*100:.1f}%)")
    print(f"{'='*100}")

    print("\nMédias das variáveis climáticas:")
    for var in variaveis_clima:
        mean_col = f'{var}_mean'
        if mean_col in cluster_data.columns:
            print(f"  {var:20s}: {cluster_data[mean_col].mean():>10.2f} (±{cluster_data[mean_col].std():>8.2f})")

# Comparação visual dos 2 clusters
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, var in enumerate(variaveis_clima):
    ax = axes[idx]

    data_plot = []
    labels_plot = []

    for i in range(n_clusters_final):
        cluster_data = df_cluster_clean[df_cluster_clean['Cluster_Final'] == i]
        data_plot.append(cluster_data[f'{var}_mean'].values)
        labels_plot.append(f'Cluster {i}')

    bp = ax.boxplot(data_plot, tick_labels=labels_plot, patch_artist=True)

    colors = ['#8dd3c7', '#fb8072']  # Cores distintas
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)

    ax.set_title(var, fontsize=12, fontweight='bold')
    ax.set_xlabel('Cluster')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
#plt.savefig(os.path.join(diretorio_completo, 'comparacao_2_clusters.jpg'), dpi=300)
plt.show()

# Salvar resultados
df_cluster_clean.to_csv(os.path.join(diretorio_completo, 'resultados_cluster_final.csv'))

# Adicionado Cluster aos atributos dos arquivos .nc

In [1]:
import xarray as xr
import pandas as pd
import os
import glob
from tqdm import tqdm

# Caminho dos arquivos
diretorio_completo = os.path.join(os.getcwd(), 'inputs', 'data', 'completo')
arquivos_nc = glob.glob(os.path.join(diretorio_completo, "point_id_*.nc"))

# Carregar resultados do clustering
resultado_cluster=os.path.join(os.getcwd(), 'inputs', 'sheets')
df_clusters = pd.read_csv(os.path.join(resultado_cluster, 'resultados_cluster.csv'), index_col='ID')

print(f"\n{'='*80}")
print("ADICIONANDO CLUSTERS COMO METADADOS NOS ARQUIVOS .NC")
print(f"{'='*80}")
print(f"\nTotal de arquivos: {len(arquivos_nc)}")
print(f"Total de IDs com cluster: {len(df_clusters)}")

# Adicionar metadados
sucessos = 0
falhas = 0

for arquivo in tqdm(arquivos_nc, desc="Atualizando metadados"):
    ponto_id = os.path.basename(arquivo).replace('point_id_', '').replace('.nc', '')

    try:
        # Verificar se o ID tem cluster atribuído
        if ponto_id not in df_clusters.index:
            print(f"\n⚠️ ID {ponto_id} não encontrado nos resultados de clustering")
            falhas += 1
            continue

        # Obter informações do cluster
        cluster_id = int(df_clusters.loc[ponto_id, 'Cluster'])

        # Abrir arquivo, adicionar metadados e fechar
        ds = xr.open_dataset(arquivo)

        # Adicionar atributos globais
        ds.attrs['cluster_id'] = cluster_id
        ds.attrs['cluster_method'] = 'Hierarchical Clustering (Ward linkage)'

        # Salvar em arquivo temporário
        arquivo_temp = arquivo + '.tmp'
        ds.to_netcdf(arquivo_temp)
        ds.close()  # Fechar dataset

        # Substituir arquivo original
        os.replace(arquivo_temp, arquivo)

        sucessos += 1

    except Exception as e:
        print(f"\n❌ Erro ao processar {ponto_id}: {e}")
        falhas += 1
        # Limpar arquivo temporário se houver erro
        if os.path.exists(arquivo + '.tmp'):
            os.remove(arquivo + '.tmp')

print(f"\n{'='*80}")
print("RESULTADO DA ATUALIZAÇÃO")
print(f"{'='*80}")
print(f"✅ Arquivos atualizados com sucesso: {sucessos}")
print(f"❌ Falhas: {falhas}")

# Verificar um arquivo como exemplo
if sucessos > 0:
    arquivo_exemplo = arquivos_nc[0]
    ponto_id_exemplo = os.path.basename(arquivo_exemplo).replace('point_id_', '').replace('.nc', '')

    print(f"\n{'='*80}")
    print("EXEMPLO DE METADADOS ADICIONADOS")
    print(f"{'='*80}")

    with xr.open_dataset(arquivo_exemplo) as ds:
        print(f"\nArquivo: {ponto_id_exemplo}.nc")
        print(f"  cluster_id: {ds.attrs.get('cluster_id', 'N/A')}")
        print(f"  cluster_method: {ds.attrs.get('cluster_method', 'N/A')}")


ADICIONANDO CLUSTERS COMO METADADOS NOS ARQUIVOS .NC

Total de arquivos: 1506
Total de IDs com cluster: 1506


Atualizando metadados: 100%|██████████| 1506/1506 [01:17<00:00, 19.38it/s]


RESULTADO DA ATUALIZAÇÃO
✅ Arquivos atualizados com sucesso: 1506
❌ Falhas: 0

EXEMPLO DE METADADOS ADICIONADOS

Arquivo: 000b2c5b.nc
  cluster_id: 1
  cluster_method: Hierarchical Clustering (Ward linkage)
  cluster_resolution: N/A
  cluster_n_groups: N/A


# Mapas com Geopandas
Observando a distribuição espacial dos clusters

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import os
import glob
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point
from tqdm import tqdm
import geodatasets

# Configurações
diretorio_completo = os.path.join(os.getcwd(), 'inputs', 'data', 'completo')
arquivos_nc = glob.glob(os.path.join(diretorio_completo, "point_id_*.nc"))
variaveis_clima = ['ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'T2M_MAX', 'T2M_MIN', 'RH2M', 'WS2M']
n_clusters_range = range(2, 11)

print(f"Total de arquivos: {len(arquivos_nc)}")

# Extrair estatísticas e coordenadas de cada arquivo
dados_agregados = []

for arquivo in tqdm(arquivos_nc, desc="Processando arquivos"):
    ponto_id = os.path.basename(arquivo).replace('point_id_', '').replace('.nc', '')

    try:
        with xr.open_dataset(arquivo) as ds:
            estatisticas = {'ID': ponto_id}

            # Extrair coordenadas dos atributos globais
            if 'latitude' in ds.attrs and 'longitude' in ds.attrs:
                estatisticas['lat'] = float(ds.attrs['latitude'])
                estatisticas['lon'] = float(ds.attrs['longitude'])

            for var in variaveis_clima:
                if var in ds.data_vars:
                    estatisticas[f'{var}_mean'] = float(ds[var].mean(skipna=True).values)
                    estatisticas[f'{var}_std'] = float(ds[var].std(skipna=True).values)
                    estatisticas[f'{var}_median'] = float(ds[var].median(skipna=True).values)
                    estatisticas[f'{var}_min'] = float(ds[var].min(skipna=True).values)
                    estatisticas[f'{var}_max'] = float(ds[var].max(skipna=True).values)

            dados_agregados.append(estatisticas)

    except Exception as e:
        print(f"Erro ao processar {ponto_id}: {e}")

# Criar DataFrame
df_cluster = pd.DataFrame(dados_agregados)
df_cluster = df_cluster.set_index('ID')

# Separar coordenadas antes de remover NaN
coordenadas = df_cluster[['lat', 'lon']].copy()

# Remover colunas de coordenadas e NaN para clustering
df_cluster_features = df_cluster.drop(columns=['lat', 'lon'], errors='ignore').dropna()

print(f"\nDados processados: {df_cluster_features.shape}")
print(f"Exemplo de IDs: {df_cluster_features.index[:5].tolist()}")

# Padronizar os dados
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster_features)

# Carregar mapa mundi
world = gpd.read_file(geodatasets.get_path('naturalearth.land'))

# Criar diretório para salvar mapas
dir_mapas = os.path.join(os.getcwd(),'output','mapas_clusters')
os.makedirs(dir_mapas, exist_ok=True)

print(f"\n{'='*80}")
print("TESTANDO DIFERENTES NÚMEROS DE CLUSTERS")
print(f"{'='*80}")

for n_clusters in n_clusters_range:
    print(f"\nProcessando {n_clusters} clusters...")

    # Aplicar clustering
    model = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
    clusters = model.fit_predict(X_scaled)

    # Adicionar clusters ao DataFrame
    df_resultado = df_cluster_features.copy()
    df_resultado['Cluster'] = clusters

    # Juntar com coordenadas
    df_resultado = df_resultado.join(coordenadas, how='inner')

    # Validar coordenadas
    df_valido = df_resultado[
        (df_resultado['lat'].between(-90, 90)) &
        (df_resultado['lon'].between(-180, 180)) &
        df_resultado[['lat', 'lon']].notna().all(axis=1)
    ].copy()

    erros = len(df_resultado) - len(df_valido)
    if erros > 0:
        print(f"  ⚠️ {erros} pontos ignorados (coordenadas inválidas)")

    # Criar GeoDataFrame
    gdf = gpd.GeoDataFrame(
        df_valido,
        geometry=[Point(lon, lat) for lon, lat in zip(df_valido['lon'], df_valido['lat'])],
        crs='EPSG:4326'
    )

    print(f"  ✓ {len(gdf)} pontos válidos para plotagem")

    # Plotar mapa
    fig, ax = plt.subplots(figsize=(20, 12))

    # Mapa base
    world.plot(ax=ax, color='lightgray', edgecolor='white', linewidth=0.5)

    # Pontos coloridos por cluster (com marcadores menores)
    gdf.plot(
        ax=ax,
        column='Cluster',
        cmap='tab10',
        markersize=5,  # Era 50, agora 15 (ajuste conforme necessário)
        alpha=0.6,  # Reduzir também a opacidade ajuda na sobreposição
        legend=True,
        categorical=True,
        legend_kwds={'title': 'Cluster ID', 'loc': 'lower left', 'fontsize': 10}
    )

    ax.set_title(f'Distribuição Espacial - {n_clusters} Clusters', fontsize=18, fontweight='bold', pad=20)
    ax.set_xlabel('Longitude', fontsize=14)
    ax.set_ylabel('Latitude', fontsize=14)
    ax.grid(True, alpha=0.3, linestyle='--')

    # Adicionar texto com estatísticas
    texto_stats = f"Total de pontos: {len(gdf)}\n"
    for cluster_id in sorted(gdf['Cluster'].unique()):
        n_pontos = len(gdf[gdf['Cluster'] == cluster_id])
        percentual = (n_pontos / len(gdf)) * 100
        texto_stats += f"Cluster {cluster_id}: {n_pontos} ({percentual:.1f}%)\n"

    ax.text(0.02, 0.98, texto_stats, transform=ax.transAxes,
            fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    plt.tight_layout()
    plt.savefig(os.path.join(dir_mapas, f'mapa_{n_clusters}_clusters.jpg'), dpi=300, bbox_inches='tight')
    plt.close()

    # Estatísticas por cluster
    print(f"\n  Distribuição dos {n_clusters} clusters:")
    for cluster_id in sorted(gdf['Cluster'].unique()):
        cluster_data = gdf[gdf['Cluster'] == cluster_id]
        print(f"    Cluster {cluster_id}: {len(cluster_data)} pontos ({len(cluster_data)/len(gdf)*100:.1f}%)")
        print(f"      Lat: {cluster_data['lat'].min():.2f}° a {cluster_data['lat'].max():.2f}°")
        print(f"      Lon: {cluster_data['lon'].min():.2f}° a {cluster_data['lon'].max():.2f}°")

    # Salvar resultados
    arquivo_resultado = os.path.join(dir_mapas, f'clusters_{n_clusters}.csv')
    df_resultado.to_csv(arquivo_resultado)
    print(f"  ✅ Mapa e resultados salvos")

print(f"\n{'='*80}")
print(f"ANÁLISE CONCLUÍDA")
print(f"{'='*80}")
print(f"\nMapas salvos em: {dir_mapas}")

# teste

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# Configurar modelos
configs = [
    {'n_clusters': 5, 'nome': '5 Clusters'},
    {'n_clusters': 6, 'nome': '6 Clusters'}
]

resultados_comparacao = []

for config in configs:
    k = config['n_clusters']

    # Treinar modelo
    model = AgglomerativeClustering(n_clusters=k, linkage='ward', compute_distances=True)
    clusters = model.fit_predict(X_scaled)

    # Calcular métricas
    sil = silhouette_score(X_scaled, clusters)
    db = davies_bouldin_score(X_scaled, clusters)
    ch = calinski_harabasz_score(X_scaled, clusters)

    resultados_comparacao.append({
        'Clusters': k,
        'Silhouette': sil,
        'Davies-Bouldin': db,
        'Calinski-Harabasz': ch
    })

    print(f"\n{'='*60}")
    print(f"{config['nome']}")
    print(f"{'='*60}")
    print(f"Silhouette Score:       {sil:.4f}")
    print(f"Davies-Bouldin Index:   {db:.4f}")
    print(f"Calinski-Harabasz:      {ch:.2f}")

    # Distribuição dos pontos
    unique, counts = np.unique(clusters, return_counts=True)
    print(f"\nDistribuição:")
    for cluster_id, count in zip(unique, counts):
        print(f"  Cluster {cluster_id}: {count} pontos ({count/len(clusters)*100:.1f}%)")

# Criar DataFrame comparativo
df_comparacao = pd.DataFrame(resultados_comparacao)
print(f"\n{'='*60}")
print("RESUMO COMPARATIVO")
print(f"{'='*60}")
print(df_comparacao.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Métricas
metricas = ['Silhouette', 'Davies-Bouldin', 'Calinski-Harabasz']
cores = ['#2ecc71', '#e74c3c', '#3498db']

for idx, metrica in enumerate(metricas):
    ax = axes[idx]
    valores = df_comparacao[metrica].values

    bars = ax.bar(['5 Clusters', '6 Clusters'], valores, color=cores[idx], alpha=0.7)

    # Adicionar valores nas barras
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontweight='bold', fontsize=11)

    ax.set_title(metrica, fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

    # Destacar melhor valor
    if metrica == 'Davies-Bouldin':
        melhor_idx = valores.argmin()  # Menor é melhor
    else:
        melhor_idx = valores.argmax()  # Maior é melhor

    bars[melhor_idx].set_edgecolor('black')
    bars[melhor_idx].set_linewidth(3)

plt.tight_layout()
#plt.savefig('comparacao_5_vs_6_clusters.jpg', dpi=300, bbox_inches='tight')
plt.show()